# INDICATION Detection — Rule-Out Context

INDICATION is the most clinically important category that medspaCy misses entirely.

**"Rule out PE"** means: *we do not yet know whether PE is present — imaging was ordered to
answer that question.* It is not the same as negation. A patient sent for CT-PA to rule out PE
may well have PE. Treating this as `is_negated = True` (the medspaCy default) is a clinical
error that downstream reasoning systems will propagate.

cwyde represents this epistemic state formally as:

> **INDICATION(X)** ≡ ¬K_clinician(X) ∧ ¬K_clinician(¬X)

The clinician neither knows X is present nor knows X is absent — it is an open question.

In [1]:
from loguru import logger
logger.disable("PyRuSH")

import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
])

## 1. Classic rule-out phrasings

medspaCy's ConText fires the negation modifier on these — `is_negated = True` — because lexicons from the early 2010s often listed 'rule out' as a negation cue. cwyde's indication detector promotes them to INDICATION.

In [2]:
ruleout_sentences = [
    "Rule out pulmonary embolism.",
    "R/O PE.",
    "Evaluate for pulmonary embolism.",
    "To assess for PE.",
    "CT-PA to evaluate for PE.",
]

col = [45, 12, 22]
headers = ["Sentence", "is_negated", "cwyde_category"]
print("".join(h.ljust(col[i]) for i, h in enumerate(headers)))
print("-" * sum(col))
for text in ruleout_sentences:
    doc = nlp(text)
    for ent in doc.ents:
        row = [text[:43], str(ent._.is_negated), ent._.cwyde_assertion_category.value]
        print("".join(v.ljust(col[i]) for i, v in enumerate(row)))

Sentence                                     is_negated  cwyde_category        
-------------------------------------------------------------------------------
Rule out pulmonary embolism.                 False       INDICATION            
R/O PE.                                      False       INDICATION            
Evaluate for pulmonary embolism.             False       INDICATION            


To assess for PE.                            False       INDICATION            
CT-PA to evaluate for PE.                    False       INDICATION            


## 2. Backfill patterns — clinical concern phrases

Some INDICATION contexts use no explicit rule-out phrase. cwyde's backfill patterns catch phrasings like "concern for" and "question of" that express the same open-question epistemic state.

In [3]:
backfill_sentences = [
    "Concern for pulmonary embolism.",
    "Question of PE given elevated D-dimer.",
    "Possible PE — imaging ordered.",
    "Worrisome for pulmonary embolism.",
]

for text in backfill_sentences:
    doc = nlp(text)
    for ent in doc.ents:
        cat = ent._.cwyde_assertion_category.value
        trace = ent._.cwyde_resolution_trace
        steps = " → ".join(f"{s['step']}:{str(s.get('cwyde','?')).split('.')[-1]}" for s in trace)
        print(f"  {cat:35s}  {text[:48]}")
        print(f"    {steps}")

  INDICATION                           Concern for pulmonary embolism.
    category_mapper:? → indication_detector:?
  INDICATION                           Question of PE given elevated D-dimer.
    category_mapper:? → indication_detector:?
  PROBABLE_EXISTENCE                   Possible PE — imaging ordered.
    category_mapper:PROBABLE_EXISTENCE
  INDICATION                           Worrisome for pulmonary embolism.
    category_mapper:PROBABLE_EXISTENCE → indication_detector:?


## 3. INDICATION vs. confirmed negation

The critical distinction: once the study is done and PE is ruled out, the report should say "No evidence of PE" — `DEFINITE_NEGATED_EXISTENCE`. cwyde preserves this distinction.

In [4]:
contrast = [
    ("INDICATION — open question",  "Rule out PE."),
    ("INDICATION — open question",  "Evaluate for PE."),
    ("NEGATED — confirmed absent",   "No evidence of PE."),
    ("NEGATED — confirmed absent",   "No PE identified."),
    ("NEGATED — probable absent",    "PE is unlikely."),
]

print(f"{'Expected':35s}  {'Got':35s}  {'OK':2s}  Sentence")
print("-" * 110)
for label, text in contrast:
    doc = nlp(text)
    for ent in doc.ents:
        got = ent._.cwyde_assertion_category.value
        ok  = "✓" if ("INDICATION" in label and got == "INDICATION") or                        ("NEGATED" in label and "NEGATED" in got or "NEGATED" in label and got == "PROBABLE_NEGATED_EXISTENCE") else "✗"
        print(f"  {label:33s}  {got:35s}  {ok}   {text}")

Expected                             Got                                  OK  Sentence
--------------------------------------------------------------------------------------------------------------


  INDICATION — open question         INDICATION                           ✓   Rule out PE.


  INDICATION — open question         INDICATION                           ✓   Evaluate for PE.
  NEGATED — confirmed absent         DEFINITE_NEGATED_EXISTENCE           ✓   No evidence of PE.


  NEGATED — confirmed absent         DEFINITE_NEGATED_EXISTENCE           ✓   No PE identified.


  NEGATED — probable absent          PROBABLE_NEGATED_EXISTENCE           ✓   PE is unlikely.


## 4. The modal formula

INDICATION is the only category not encoded as `RankedBelief`. It uses a compound epistemic formula: the clinician neither knows X nor knows ¬X.

In [5]:
import json
from cwyde.formal.translator import category_to_formula
from cwyde.categories import AssertionCategory

f_indication = category_to_formula(AssertionCategory.INDICATION, "pe")
f_definite   = category_to_formula(AssertionCategory.DEFINITE_EXISTENCE, "pe")
f_negated    = category_to_formula(AssertionCategory.DEFINITE_NEGATED_EXISTENCE, "pe")

print("INDICATION formula (¬K_clinician(PE) ∧ ¬K_clinician(¬PE)):")
print(json.dumps(f_indication.to_tree_json(), indent=2))
print()
print("DEFINITE_EXISTENCE: ", f_definite)
print("DEFINITE_NEGATED:   ", f_negated)

INDICATION formula (¬K_clinician(PE) ∧ ¬K_clinician(¬PE)):
{
  "type": "and",
  "left": {
    "type": "not",
    "operand": {
      "type": "knowledge",
      "agent": "clinician",
      "operand": {
        "type": "atom",
        "name": "pe"
      }
    }
  },
  "right": {
    "type": "not",
    "operand": {
      "type": "knowledge",
      "agent": "clinician",
      "operand": {
        "type": "not",
        "operand": {
          "type": "atom",
          "name": "pe"
        }
      }
    }
  }
}

DEFINITE_EXISTENCE:  RankedBelief(agent='clinician', rank=2, operand=Atom(name='pe'))
DEFINITE_NEGATED:    RankedBelief(agent='clinician', rank=-2, operand=Atom(name='pe'))


## 5. INDICATION is compatible with discovery

INDICATION is an *open* epistemic state — subsequent assertions about the same finding
can resolve it. `gamen-validate` treats INDICATION as consistent with both
DEFINITE_EXISTENCE and DEFINITE_NEGATED_EXISTENCE, because neither contradicts
"we didn't yet know."

In [6]:
from cwyde_haskell_bridge import GamenBridge
bridge = GamenBridge()

pairs = [
    ("INDICATION + DEFINITE_EXISTENCE (finding confirmed)",
     AssertionCategory.INDICATION, AssertionCategory.DEFINITE_EXISTENCE),
    ("INDICATION + DEFINITE_NEGATED_EXISTENCE (finding ruled out)",
     AssertionCategory.INDICATION, AssertionCategory.DEFINITE_NEGATED_EXISTENCE),
    ("DEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (contradiction)",
     AssertionCategory.DEFINITE_EXISTENCE, AssertionCategory.DEFINITE_NEGATED_EXISTENCE),
]

for label, cat_a, cat_b in pairs:
    f_a = category_to_formula(cat_a, "pe")
    f_b = category_to_formula(cat_b, "pe")
    result = bridge.check_consistency([f_a, f_b])
    print(f"  {'consistent' if result.consistent else 'INCONSISTENT':13s}  {label}")

  consistent     INDICATION + DEFINITE_EXISTENCE (finding confirmed)


  consistent     INDICATION + DEFINITE_NEGATED_EXISTENCE (finding ruled out)
  INCONSISTENT   DEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (contradiction)


## Takeaway

INDICATION is not negation — it is an open epistemic question. The table above confirms:
- A finding under investigation (INDICATION) is consistent with either outcome.
- A finding simultaneously asserted present AND absent is a logical contradiction.

This distinction matters for clinical decision support: a "rule out PE" order should not
suppress PE-related alerts the way a confirmed "no PE" would.